# Prepare the Twitter dataset

This notebook discovers every Twitter JSONL.GZ file in data/twitter and processes batches in filename order. It divides MAX_TWEETS evenly across batches and uses the selected sampling strategy for each batch. Add future daily archives to data/twitter; the notebook discovers them automatically.

"first" is fast but samples early rows. "reservoir" scans each full archive for a random sample. "balanced" samples 1 in 10 rows to estimate label counts, then scans each archive again to sample balanced groups where data allows. Its default field is retweet-count bands. The selected tweet table is cached separately, so changing the embedding model reuses the sample when the source files and sampling settings still match.

## 1. Install dependencies

Run once per environment. The first embedding run also downloads Google's model weights from Hugging Face.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
%pip install -e "{PROJECT_ROOT}[notebook,preprocessing]"


Obtaining file:///Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imste (pyproject.toml) ... done
  Created wheel for imste: filename=imste-0.8.11-0.editable-py3-none-any.whl size=15702 sha256=921c5185a129cd4a30680f25ab98e4288a89b31b7d79943f941b0a3d130534e5
  Stored in directory: /private/var/folders/wr/3lmv1jtx3ng07d2c77xprblh0000gp/T/pip-ephem-wheel-cache-wi2b6o7f/wheels/a1/4c/5f/1d0dd04bf55afbb4d710bd8563f6fca0bdda1e32976365f2b3
Successfully built imste
  Attempting uninstall: imste
    Found existing installation: imste 0.8.11
    Uninstalling imste-0.8.11:
      Successfully uninstalled imste-0.8.11
Note: you may need to restart the kernel to use updated packages.


## 2. Configure the sample and build embeddings

Set LANGUAGE to a Twitter language code (for example en) or None to accept all languages. MAX_TWEETS is the requested total across all batches. The notebook divides it evenly, giving any remainder to the earliest batches.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "notebooks" / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
DATA_DIR = PROJECT_ROOT / "data" / "twitter"
SOURCE_PATHS = sorted(DATA_DIR.glob("*.json.gz"), key=lambda path: path.name)
if not SOURCE_PATHS:
    raise FileNotFoundError(f"No Twitter JSON.GZ batches found in {DATA_DIR}")
print("Batch order:")
for path in SOURCE_PATHS:
    print(f"  - {path.name} ({path.stat().st_size / 1024**3:.2f} GiB compressed)")

Batch order:
  - 2014-12-01.world.json.gz (3.66 GiB compressed)
  - 2014-12-02.world.json.gz (3.54 GiB compressed)
  - 2014-12-03.world.json.gz (3.69 GiB compressed)


In [ ]:
import pandas as pd
from twitter_dataset import prepare_twitter_dataset
MAX_TWEETS = 2_000_000
SAMPLE_STRATEGY = "balanced"  # first, reservoir, or balanced
BALANCE_BY = "language"  # retweet_count, language, retweet, hour, or weekday
BALANCE_COUNT_STRIDE = 10  # estimate label counts from every 10th source row
RANDOM_STATE = 42
LANGUAGE = None                 # e.g. "en", or None for all languages
EMBEDDING_MODEL = "google/embeddinggemma-2"
BATCH_SIZE = 64
FORCE_REBUILD = False

tweets, X, manifest = prepare_twitter_dataset(
    DATA_DIR, SOURCE_PATHS, max_tweets=MAX_TWEETS,
    random_state=RANDOM_STATE, language=LANGUAGE,
    embedding_model=EMBEDDING_MODEL, batch_size=BATCH_SIZE,
    sample_strategy=SAMPLE_STRATEGY, balance_by=BALANCE_BY,
    balance_count_stride=BALANCE_COUNT_STRIDE,
    force_rebuild=FORCE_REBUILD,
)
print(f"Prepared {len(tweets):,} tweets from {manifest['n_batches']} batches × {X.shape[1]:,} features")
print(f"Vector matrix: {X.nbytes / 1024**3:.2f} GiB")
print("Batch order:", manifest["batch_order"])
print("Sampling strategy:", manifest["sample_strategy"])
print("Balanced by:", manifest.get("balance_by"))
print("Count sample stride:", manifest.get("balance_count_stride"))
print("Batch quotas:", manifest["batch_quotas"])
print("Dataset ID:", manifest["dataset_id"])
if SAMPLE_STRATEGY == "balanced" and BALANCE_BY == "retweet_count":
    retweet_counts = pd.to_numeric(tweets["retweet_count"], errors="coerce").fillna(0).clip(lower=0)
    bands = pd.cut(
        retweet_counts,
        bins=[-0.5, 0.5, 1.5, 5.5, 20.5, 100.5, float("inf")],
        labels=["0", "1", "2–5", "6–20", "21–100", ">100"],
    )
    print("Sampled counts by retweet-count band:")
    print(bands.value_counts(sort=False))
elif SAMPLE_STRATEGY == "balanced" and BALANCE_BY == "language":
    print("Sampled counts by language:")
    print(tweets["lang"].fillna("unknown").value_counts().sort_index())
tweets.head(10)

Reusing cached balanced sample (1,988,195 tweets); archive scans skipped.
Embedding 1,988,195 tweets from 3 ordered batches with sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2…


W1123 17:13:12.064000 87572 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using embedding prompt: query


Batches:   0%|          | 0/31066 [00:00<?, ?it/s]

Saved 1,988,195 tweets × 384 features from 3 batches in /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/twitter/cache.
Prepared 1,988,195 tweets from 3 batches × 384 features
Vector matrix: 2.84 GiB
Batch order: ['2014-12-01.world.json.gz', '2014-12-02.world.json.gz', '2014-12-03.world.json.gz']
Sampling strategy: balanced
Balanced by: language
Count sample stride: 10
Batch quotas: {'2014-12-01.world.json.gz': 666667, '2014-12-02.world.json.gz': 666667, '2014-12-03.world.json.gz': 666666}
Dataset ID: 7dc37cf51c52345bc93f
Sampled counts by language:
lang
am         7
ar     80978
bg     15887
bn      1711
bo        23
       ...  
uk     21784
und    80966
ur      7657
vi     18140
zh     19993
Name: count, Length: 66, dtype: int64


,source_batch,tweet_id,created_at,lang,text,hashtags,top_hashtag,retweet_count,favorite_count,is_retweet
0,2014-12-01.world.json.gz,539192145358630912,Sun Nov 30 22:59:59 +0000 2014,und,.,[],(no hashtag),0,0,False
1,2014-12-01.world.json.gz,539192146403004416,Sun Nov 30 23:00:00 +0000 2014,in,@zouisorude AMRO,[],(no hashtag),0,0,False
2,2014-12-01.world.json.gz,539192146688212992,Sun Nov 30 23:00:00 +0000 2014,ar,@alanezi94 @Nayefal3jmi اي طبعا 😂 ابو مانع هو...,[],(no hashtag),0,0,False
3,2014-12-01.world.json.gz,539192146834624512,Sun Nov 30 23:00:00 +0000 2014,ko,พี่บีหล่อเกินไปㅠㅠ,[],(no hashtag),0,0,False
4,2014-12-01.world.json.gz,539192147199549440,Sun Nov 30 23:00:00 +0000 2014,en,"In a perfect world, that ass hat Jameis Winsto...",[],(no hashtag),0,0,False
5,2014-12-01.world.json.gz,539192147824893952,Sun Nov 30 23:00:00 +0000 2014,de,bäm bäm bäm bäm bäm bäm bäm bäm bäm bäm bäm bäm,[],(no hashtag),0,0,False
6,2014-12-01.world.json.gz,539192147875213313,Sun Nov 30 23:00:00 +0000 2014,ko,에밀레 에밀레 에밀레 에밀레 에밀레 에밀레 에밀레 에밀레,[],(no hashtag),0,0,False
7,2014-12-01.world.json.gz,539192148625997824,Sun Nov 30 23:00:00 +0000 2014,tr,Yine kalacak yerim yok aklında kalabilirmiyim :)),[],(no hashtag),0,0,False
8,2014-12-01.world.json.gz,539192148525350912,Sun Nov 30 23:00:00 +0000 2014,fa,اون عکس چیه توکیفت؟\n-عکس نیست چکامه,[],(no hashtag),0,0,False
9,2014-12-01.world.json.gz,539192148986724354,Sun Nov 30 23:00:00 +0000 2014,nl,Die eerste 5min. in u bed is om te bevriezen,[],(no hashtag),0,0,False
